# RNA queue extension, 2026-11-01


# RNA queue extension (2026-11-01)

Einar asked to finish all downloads. The coverage ledger listed 109 catalogue RNA studies that had never been queued. This notebook re-reads the review and staging receipts on the NAS and shows what was added, what was left out and why. It changes nothing.

In [1]:
import csv, json, collections
AUDIT = '/nas_mnt/carruthers/Einar/parascape_omics/processing_audit/unplanned_rna_study_review'
studies = list(csv.DictReader(open(AUDIT + '/studies.tsv'), delimiter='\t'))
runs = list(csv.DictReader(open(AUDIT + '/runs.tsv'), delimiter='\t'))
len(studies), len(runs), sum(1 for s in studies if s.get('error'))

(109, 18475, 0)

## Why each run would or would not be queued

Decisions use the existing `omics_quant.plan()` filter: Toxoplasma scientific name and an RNA-Seq/OTHER/Ribo-Seq library strategy. Bytes are ENA FASTQ sizes.

In [2]:
dec = collections.Counter(r['decision'] for r in runs)
{k: (v, round(sum(int(r['fastq_bytes'] or 0) for r in runs if r['decision'] == k) / 1e12, 2)) for k, v in dec.items()}

{'other_strategy': (134, 0.33), 'host_scientific_name': (748, 1.89), 'keep_current_filter': (17577, 4.88), 'already_planned': (12, 0.04), 'no_fastq': (4, 0.0)}

## Per-cell collections are not bulk RNA-seq

Two studies have far more runs than GEO samples (one run per cell). Their authors deposited AnnData matrices, already prepared by the sparse pipeline, so they are excluded from bulk salmon/kallisto quantification.

In [3]:
[(s['study'], s['keep_current_filter'], s['catalogue_samples']) for s in studies if s['study'] in ('GSE145800', 'GSE145080')]

[('GSE145080', '5499', '8.0'), ('GSE145800', '11419', '23.0')]

## Host-labelled runs

Runs named after a host species are often dual RNA-seq of infected tissue. Only hosts whose transcripts are in the competitive ME49_host reference (human, mouse) were queued; others would let conserved host reads count as parasite.

In [4]:
hosts = collections.Counter(r['scientific_name'] for r in runs if r['decision'] == 'host_scientific_name' and r['study'] not in ('GSE145800', 'GSE145080'))
hosts.most_common()

[('Homo sapiens', 325), ('Mus musculus', 290), ('', 80), ('Bos taurus', 28), ('Mesocricetus auratus', 9), ('Sus scrofa', 6), ('Neospora caninum', 6), ('Felis catus', 3), ('Plasmodium falciparum 3D7', 1)]

## What was appended

Both staging receipts pin the exact rows by SHA-256; the operations log records each append and its backup.

In [5]:
bulk = json.load(open(AUDIT + '/staged-receipt.json')); host = json.load(open(AUDIT + '/staged-host-labelled-receipt.json'))
ops = [json.loads(l) for l in open(AUDIT + '/operations.jsonl')]
[(r['staged_rows'], round(r['fastq_bytes'] / 1e12, 2), r['staged_sha256'][:16]) for r in (bulk, host)], [(o['added_rows'], o['planned_runs']) for o in ops]

([(675, 3.44, 'c3cb0414a8763ae7'), (620, 1.38, '305ebfc66e7dc640')], [(675, 1686), (620, 2306)])

## Limits

Queued runs are planned downloads, not verified quantifications. Run counts are associations; 12 runs are shared with studies already planned. Organism names are ENA declarations, not verified biology.